In [1]:
import cv2
import numpy as np

def create_checkerboard(
    rows=7,           # 세로 사각형 개수 (OpenCV 코너 기준 6개 -> 사각형은 최소 7개 필요)
    cols=10,          # 가로 사각형 개수 (OpenCV 코너 기준 9개 -> 사각형은 최소 10개 필요)
    square_size_mm=25, # 사각형 한 변의 길이 (mm)
    dpi=300            # 인쇄 해상도 (보통 300 권장)
):
    # 1. mm -> pixel 변환
    # 1 inch = 25.4 mm
    px_per_mm = dpi / 25.4
    square_size_px = int(square_size_mm * px_per_mm)
    
    # 2. 전체 이미지 크기 계산
    width = cols * square_size_px
    height = rows * square_size_px
    
    # 3. 빈 이미지 생성 (흰색)
    board = np.ones((height, width), dtype=np.uint8) * 255
    
    # 4. 검은색 사각형 그리기
    for y in range(rows):
        for x in range(cols):
            # 행+열 인덱스 합이 홀수면 검은색
            if (x + y) % 2 == 1:
                start_x = x * square_size_px
                start_y = y * square_size_px
                end_x = start_x + square_size_px
                end_y = start_y + square_size_px
                
                # 검은색(0)으로 채우기
                board[start_y:end_y, start_x:end_x] = 0

    # 5. 여백(Margin) 추가 (인식률 향상용)
    margin_mm = 20 # 20mm 여백
    margin_px = int(margin_mm * px_per_mm)
    
    # cv2.copyMakeBorder로 흰색 여백 추가
    board_with_margin = cv2.copyMakeBorder(
        board, 
        margin_px, margin_px, margin_px, margin_px, 
        cv2.BORDER_CONSTANT, 
        value=255
    )

    return board_with_margin

# ==========================================
# 실행 설정
# ==========================================
# 주의: 이전에 쓰던 코드의 CHECKERBOARD=(9,6)을 쓰려면
# 사각형 개수는 그것보다 +1씩 커야 합니다. (가로 10칸, 세로 7칸)
img = create_checkerboard(rows=7, cols=10, square_size_mm=25)

# 이미지 저장
filename = "checkerboard_A4_print.png"
cv2.imwrite(filename, img)

print(f"파일이 생성되었습니다: {filename}")
print("인쇄할 때 반드시 '배율: 100%' 또는 '실제 크기'로 설정하세요!")

파일이 생성되었습니다: checkerboard_A4_print.png
인쇄할 때 반드시 '배율: 100%' 또는 '실제 크기'로 설정하세요!


In [2]:
import pyrealsense2 as rs
import numpy as np
import cv2
import time

# ==========================================
# [설정] 체스보드 규격 (사용하는 종이에 맞춰 수정 필수!)
# ==========================================
# 체스보드의 "내부 교차점" 개수입니다. (사각형 개수 아님!)
# 예: 가로 10칸, 세로 7칸짜리 보드라면 -> (9, 6) 입력
CHECKERBOARD = (9, 6) 

# 체스보드 한 칸의 한 변 길이 (mm 단위) - 자로 재서 정확히 입력하세요.
SQUARE_SIZE_MM = 23.0 

# ==========================================

def run_calibration():
    # 1. 3D 공간상의 체스보드 좌표 생성 (0,0,0), (1,0,0), (2,0,0) ...
    objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
    objp = objp * SQUARE_SIZE_MM

    # 3D 점(실제 세계)과 2D 점(이미지)을 저장할 리스트
    objpoints = [] # 3d point in real world space
    imgpoints = [] # 2d points in image plane.

    # RealSense 초기화
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    
    print("-------------------------------------------------------")
    print(f"체스보드({CHECKERBOARD})를 준비하세요.")
    print(" 'c' 키: 사진 캡처 (최소 15장 권장)")
    print(" 'q' 키: 캘리브레이션 시작 및 종료")
    print("-------------------------------------------------------")

    pipeline.start(config)
    
    cap_count = 0

    try:
        while True:
            frames = pipeline.wait_for_frames()
            color_frame = frames.get_color_frame()
            if not color_frame: continue

            img = np.asanyarray(color_frame.get_data())
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            vis = img.copy()

            # 체스보드 코너 찾기
            ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

            # 찾았으면 그리기
            if ret == True:
                cv2.drawChessboardCorners(vis, CHECKERBOARD, corners, ret)
                cv2.putText(vis, "Ready to Capture!", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            else:
                cv2.putText(vis, "Show Chessboard...", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            cv2.putText(vis, f"Captured: {cap_count}", (10, 450), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)
            cv2.imshow('RealSense Calibration', vis)

            key = cv2.waitKey(1)

            # 'c' 키를 누르고 체스보드가 인식된 상태라면 저장
            if key == ord('c'):
                if ret == True:
                    objpoints.append(objp)
                    
                    # 코너 좌표 정밀 보정 (Sub-pixel)
                    corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), 
                                                criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))
                    imgpoints.append(corners2)
                    
                    cap_count += 1
                    print(f"[{cap_count}] 이미지 캡처 완료!")
                    # 깜빡임 효과
                    cv2.imshow('RealSense Calibration', np.ones_like(vis)*255)
                    cv2.waitKey(50)
                else:
                    print("체스보드가 인식되지 않았습니다. 각도를 조절하세요.")

            # 'q' 키를 누르면 캘리브레이션 계산 시작
            if key == ord('q'):
                if cap_count < 10:
                    print("데이터가 너무 적습니다. 최소 10장 이상 찍어주세요.")
                else:
                    print("\n캘리브레이션 계산 중... 잠시만 기다리세요...")
                    break

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()

    if cap_count >= 10:
        # 캘리브레이션 실행!
        ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

        print("\n==============================================")
        print("       >>> 캘리브레이션 결과 (복사해서 쓰세요) <<<")
        print("==============================================")
        print(f"RMS Error (낮을수록 좋음, 0.1~0.5 목표): {ret:.4f}")
        print("\n1. Camera Matrix (intrinsics):")
        print("np.array([")
        print(f"  [{mtx[0][0]:.5f}, {mtx[0][1]:.5f}, {mtx[0][2]:.5f}],")
        print(f"  [{mtx[1][0]:.5f}, {mtx[1][1]:.5f}, {mtx[1][2]:.5f}],")
        print(f"  [{mtx[2][0]:.5f}, {mtx[2][1]:.5f}, {mtx[2][2]:.5f}]")
        print("])")
        
        print("\n2. Distortion Coefficients (k1, k2, p1, p2, k3):")
        print(f"dist_coeffs = np.array([{dist[0][0]:.5f}, {dist[0][1]:.5f}, {dist[0][2]:.5f}, {dist[0][3]:.5f}, {dist[0][4]:.5f}])")
        print("==============================================\n")

        # 결과 검증 (Undistort 테스트)
        h, w = img.shape[:2]
        newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))
        dst = cv2.undistort(img, mtx, dist, None, newcameramtx)
        
        cv2.imshow('Original', img)
        cv2.imshow('Undistorted Result (Check straight lines)', dst)
        print("보정 전/후 이미지를 확인하고 아무 키나 누르면 종료합니다.")
        cv2.waitKey(0)
        cv2.destroyAllWindows()

if __name__ == "__main__":
    run_calibration()

-------------------------------------------------------
체스보드((9, 6))를 준비하세요.
 'c' 키: 사진 캡처 (최소 15장 권장)
 'q' 키: 캘리브레이션 시작 및 종료
-------------------------------------------------------
[1] 이미지 캡처 완료!
[2] 이미지 캡처 완료!
[3] 이미지 캡처 완료!
[4] 이미지 캡처 완료!
[5] 이미지 캡처 완료!
[6] 이미지 캡처 완료!
[7] 이미지 캡처 완료!
[8] 이미지 캡처 완료!
[9] 이미지 캡처 완료!
[10] 이미지 캡처 완료!
[11] 이미지 캡처 완료!
[12] 이미지 캡처 완료!
[13] 이미지 캡처 완료!
[14] 이미지 캡처 완료!
[15] 이미지 캡처 완료!
[16] 이미지 캡처 완료!
[17] 이미지 캡처 완료!
[18] 이미지 캡처 완료!
[19] 이미지 캡처 완료!
[20] 이미지 캡처 완료!
[21] 이미지 캡처 완료!
[22] 이미지 캡처 완료!
[23] 이미지 캡처 완료!
[24] 이미지 캡처 완료!
[25] 이미지 캡처 완료!
[26] 이미지 캡처 완료!
[27] 이미지 캡처 완료!
[28] 이미지 캡처 완료!
[29] 이미지 캡처 완료!
[30] 이미지 캡처 완료!
[31] 이미지 캡처 완료!
[32] 이미지 캡처 완료!
[33] 이미지 캡처 완료!
[34] 이미지 캡처 완료!

캘리브레이션 계산 중... 잠시만 기다리세요...

       >>> 캘리브레이션 결과 (복사해서 쓰세요) <<<
RMS Error (낮을수록 좋음, 0.1~0.5 목표): 0.1836

1. Camera Matrix (intrinsics):
np.array([
  [404.86878, 0.00000, 328.17132],
  [0.00000, 406.28598, 226.54365],
  [0.00000, 0.00000, 1.00000]
])

2. Distortion Coefficients (k1